In [1]:
import sys, json
sys.path.append("../backend")
import pandas as pd, numpy as np, joblib
from features import extract_features

# held-out REAL test set: never used for training
test_phish = [l.strip() for l in open("../data/openphish.txt") if l.strip()][:500]
tr = pd.read_csv("../data/top-1m.csv", header=None, names=["rank", "domain"])
test_legit = ["https://" + d for d in tr["domain"].head(500)]

# fresh phishing for training (deduplicated, test URLs removed)
with open("../data/phishing_db.txt", errors="ignore") as f:
    fresh = [l.strip() for l in f if l.strip().startswith("http")]
fresh = [u for u in dict.fromkeys(fresh) if u not in set(test_phish)]
fresh = pd.Series(fresh).sample(n=min(30000, len(fresh)), random_state=1).tolist()

# old dataset: phishing + legit URLs that have paths
old = pd.read_csv("../data/phishing_site_urls.csv").drop_duplicates()
old_bad = old[old.Label == "bad"].URL.sample(30000, random_state=1).tolist()
old_good = old[old.Label == "good"].URL.sample(40000, random_state=1).tolist()

# legit homepages from Tranco ranks 501-20000 (test used the top 500)
home = tr["domain"].iloc[500:20000].tolist()
home_urls = ["https://" + d for d in home] + ["https://www." + d for d in home[:10000]]

urls = fresh + old_bad + old_good + home_urls
labels = [1] * (len(fresh) + len(old_bad)) + [0] * (len(old_good) + len(home_urls))
print(len(fresh), "fresh phishing |", len(old_bad), "old phishing |",
      len(old_good), "old legit |", len(home_urls), "legit homepages")

feat = pd.DataFrame([extract_features(u) for u in urls])
feat["label"] = labels
feat.to_csv("../data/features_v2.csv", index=False)
print(feat.shape)

30000 fresh phishing | 30000 old phishing | 40000 old legit | 29500 legit homepages
(129500, 29)


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X, y = feat.drop(columns=["label"]), feat["label"]
cols = list(X.columns)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                            class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)

p_val = rf.predict_proba(X_val)[:, 1]
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pr = (p_val >= t).astype(int)
    tp = ((pr == 1) & (y_val == 1)).sum(); fp = ((pr == 1) & (y_val == 0)).sum()
    print(f"val threshold {t}: precision {tp/max(tp+fp,1):.3f} | recall {tp/(y_val==1).sum():.3f}")

val threshold 0.3: precision 0.835 | recall 0.956
val threshold 0.4: precision 0.879 | recall 0.937
val threshold 0.5: precision 0.912 | recall 0.909
val threshold 0.6: precision 0.941 | recall 0.871
val threshold 0.7: precision 0.964 | recall 0.819


In [3]:
THRESHOLD = 0.7
Xt = pd.DataFrame([extract_features(u) for u in test_phish + test_legit])[cols]
yt = [1] * len(test_phish) + [0] * len(test_legit)
pt = (rf.predict_proba(Xt)[:, 1] >= THRESHOLD).astype(int)

print(classification_report(yt, pt, target_names=["Legitimate", "Phishing"]))
print(confusion_matrix(yt, pt))

joblib.dump(rf, "../backend/model_v2.pkl")
joblib.dump(cols, "../backend/feature_columns_v2.pkl")

              precision    recall  f1-score   support

  Legitimate       0.83      0.99      0.90       500
    Phishing       0.98      0.66      0.79       300

    accuracy                           0.87       800
   macro avg       0.90      0.83      0.85       800
weighted avg       0.89      0.87      0.86       800

[[496   4]
 [102 198]]


['../backend/feature_columns_v2.pkl']

In [6]:
import json
json.dump({"threshold": 0.7}, open("../backend/threshold.json", "w"))